In [1]:
# Imports
import os
import json
import pandas as pd
from openai import OpenAI
from tqdm.auto import tqdm  # Progress bar

In [2]:
# ============================================================
# CONFIGURATION - LAYER 3
# ============================================================

API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
PROMPT_FILE = "failure_summarizer_layer2.txt"
CLUSTER_DIR = "./cluster_second_round"
OUTPUT_DIR = "results/cluster_analysis_layer3"
MAX_WORKERS = 10  # Adjust based on API rate limits

# Cluster selection - ADJUST THIS LINE
# Examples (uncomment one):
cluster_index = slice(None)        # All clusters
# cluster_index = slice(0, 1)      # Only cluster_0.json
# cluster_index = slice(5, 10)     # cluster_5.json to cluster_9.json
# cluster_index = slice(None, 80)  # cluster_0.json to cluster_79.json
# cluster_index = slice(10, 20)    # cluster_10.json to cluster_19.json

In [3]:
# Load cluster files
import glob

all_cluster_files = sorted(glob.glob(os.path.join(CLUSTER_DIR, "round2cluster_*.json")), 
                           key=lambda x: int(os.path.basename(x).replace('round2cluster_', '').replace('.json', '')))

# Apply cluster selection
cluster_files = all_cluster_files[cluster_index]
print(f"✓ Found {len(all_cluster_files)} total cluster files")
print(f"✓ Selected {len(cluster_files)} clusters to process")

# Load selected clusters
clusters = []
for file_path in cluster_files:
    with open(file_path, 'r', encoding='utf-8') as f:
        cluster_data = json.load(f)
        cluster_label = int(os.path.basename(file_path).replace('round2cluster_', '').replace('.json', ''))
        clusters.append({
            'cluster_label': cluster_label,
            'file_path': file_path,
            'data': cluster_data,
            'size': len(cluster_data)
        })

print(f"✓ Loaded {len(clusters)} clusters")
if len(clusters) > 0:
    print(f"  Cluster range: {min(c['cluster_label'] for c in clusters)} - {max(c['cluster_label'] for c in clusters)}")
    print(f"  Size range: {min(c['size'] for c in clusters)} - {max(c['size'] for c in clusters)} items")

✓ Found 10 total cluster files
✓ Selected 10 clusters to process
✓ Loaded 10 clusters
  Cluster range: 0 - 9
  Size range: 1 - 16 items


In [4]:
# Load prompt template
with open(PROMPT_FILE, 'r', encoding='utf-8') as f:
    prompt_template = f.read()

print(prompt_template)

You are summarizing a cluster of reasons why an AI model's answer to Theory of Mind questions differs from the human answer. Each reason corresponds to one question.

**Task:**

Carefully read each reason and analyze step by step how the reasons in this cluster share common patterns. Then summarize your analysis in exactly 3 sentences and 90+-10 words.

**Reasons in this cluster:**

{reasons_list}



In [5]:
# Initialize OpenAI client
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=API_KEY
)

def call_gpt51_summarizer(client, prompt):
    """Call Gemini-3-Pro without reasoning"""
    response = client.chat.completions.create(
        model="openai/gpt-5.1",
        messages=[{"role": "user", "content": prompt}], 
        temperature=0.0,
        max_tokens=10240
    )
    # print(response)
    message = response.choices[0].message
    content = message.content 
    print(content)
    
    return content

print("✓ Client initialized")

✓ Client initialized


In [6]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def process_single_cluster(cluster_info):
    """Process a single cluster"""
    cluster_label = cluster_info['cluster_label']
    cluster_data = cluster_info['data']
    
    # Create thread-specific client
    thread_client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=API_KEY
    )
    
    # Build reasons list from all items in cluster
    # Layer 3: use 'cluster_summary' field from layer 2 results
    reasons_list = []
    for i, item in enumerate(cluster_data, 1):
        reason_text = f"{i}. {item['cluster_summary']}"
        reasons_list.append(reason_text)
    
    # Format prompt with all reasons
    reasons_text = "\n\n".join(reasons_list)
    prompt = prompt_template.format(reasons_list=reasons_text)
    
    try:
        cluster_summary = call_gpt51_summarizer(thread_client, prompt)
        
        result = {
            'cluster_label': cluster_label,
            'cluster_size': len(cluster_data),
            'cluster_summary': cluster_summary,
            'reasons': [item['cluster_summary'] for item in cluster_data],
            'round1_cluster_labels': [item['round1_cluster_label'] for item in cluster_data]
        }
        return cluster_label, result, None
        
    except Exception as e:
        print(f"\n✗ Error at cluster {cluster_label}: {str(e)[:100]}")
        result = {
            'cluster_label': cluster_label,
            'cluster_size': len(cluster_data),
            'cluster_summary': f"ERROR: {str(e)}",
            'reasons': [item['cluster_summary'] for item in cluster_data],
            'round1_cluster_labels': [item['round1_cluster_label'] for item in cluster_data]
        }
        return cluster_label, result, str(e)

# Process clusters with multithreading
results = []

print(f"Processing {len(clusters)} clusters with {MAX_WORKERS} threads...")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    # Submit all tasks
    future_to_cluster = {
        executor.submit(process_single_cluster, cluster): cluster['cluster_label']
        for cluster in clusters
    }
    
    # Process completed tasks with progress bar
    for future in tqdm(as_completed(future_to_cluster), 
                      total=len(clusters), 
                      desc="Processing clusters"):
        cluster_label, result, error = future.result()
        results.append(result)

print(f"\n✓ Completed processing {len(results)} clusters")

Processing 10 clusters with 10 threads...


Processing clusters:   0%|          | 0/10 [00:00<?, ?it/s]

These reasons all describe a shared failure in mapping vague quantifiers to numbers: the model treats terms like “most,” “almost no,” “few,” and “some” as fixed, often extreme values instead of flexible ranges. It overweights explicit numerals and simple subtraction, collapses multi-category or probabilistic setups into binary splits, and ignores narrative timing, implicit comparisons, and the requirement that minority categories remain small-but-nonzero. Humans, by contrast, integrate all constraints—totals, verbal hedges, expectations, and sampling uncertainty—yielding arithmetically consistent yet pragmatically moderate distributions where majority terms are strong but not maximized and “almost none” still leaves a remainder.
Across this cluster, the model’s answers systematically diverge from the key because it adopts a different interpretive frame, often privileging either sophisticated social-pragmatic reasoning or strict textual literalism over the test’s intended commonsense sh

In [7]:
# Convert to DataFrame and sort by cluster label
results_df = pd.DataFrame(results)

# Sort by cluster label to maintain order
if 'cluster_label' in results_df.columns:
    results_df = results_df.sort_values('cluster_label').reset_index(drop=True)

print(f"Results: {results_df.shape}")
print(f"\nWith summary: {results_df['cluster_summary'].notna().sum()}/{len(results_df)}")
print(f"Errors: {results_df['cluster_summary'].str.startswith('ERROR').sum()}")

Results: (10, 5)

With summary: 10/10
Errors: 0


In [8]:
results_df

,cluster_label,cluster_size,cluster_summary,reasons,round1_cluster_labels
0,0,2,Both reasons describe the model over‑relying o...,"[Across these cases, the model leans on stereo...","[29, 43]"
1,1,8,"Across these reasons, the model resolves under...","[Across these items, the model repeatedly priv...","[4, 28, 42, 44, 49, 65, 69, 73]"
2,2,16,"Across the cluster, the model over-relies on l...","[Across these items, the model leans on litera...","[5, 13, 19, 21, 23, 25, 31, 35, 48, 51, 52, 53..."
3,3,10,"Taken together, these reasons portray a model ...","[Across these items, the model repeatedly reli...","[3, 11, 15, 16, 34, 41, 47, 61, 62, 71]"
4,4,13,These reasons all describe a shared failure in...,"[Across these reasons, the core pattern is how...","[2, 7, 18, 26, 27, 33, 37, 38, 39, 57, 68, 72,..."
5,5,8,"Across the cluster, the model applies a rigid,...","[Across the cluster, disagreements stem from m...","[14, 17, 22, 30, 46, 50, 58, 70]"
6,6,12,"Across the cluster, the model does shallow pat...","[Across these reasons, the model relies on a s...","[0, 9, 20, 24, 32, 36, 40, 59, 60, 66, 67, 78]"
7,7,5,"Across the reasons, the model is portrayed as ...","[Across these items, the model relies on gener...","[6, 8, 12, 45, 54]"
8,8,1,"Across this cluster, the model repeatedly igno...","[Across these reasons, the core pattern is tha...",[64]
9,9,5,"Across this cluster, the model’s answers syste...",[This cluster reflects a pattern where the mod...,"[1, 10, 56, 63, 76]"


In [9]:
# Statistics
print(f"Total clusters analyzed: {len(results_df)}")
print(f"With summaries: {results_df['cluster_summary'].notna().sum()}")
print(f"With errors: {results_df['cluster_summary'].str.startswith('ERROR').sum()}")
print(f"Total failures covered: {results_df['cluster_size'].sum()}")
print(f"Avg cluster size: {results_df['cluster_size'].mean():.1f}")
print(f"Cluster size range: {results_df['cluster_size'].min()} - {results_df['cluster_size'].max()}")
# Compute word counts
word_counts = results_df['cluster_summary'].str.split().str.len()
print(f"Avg summary length: {word_counts.mean():.1f} words")
print(f"Max summary length: {word_counts.max()} words")
print(f"Min summary length: {word_counts.min()} words")

Total clusters analyzed: 10
With summaries: 10
With errors: 0
Total failures covered: 80
Avg cluster size: 8.0
Cluster size range: 1 - 16
Avg summary length: 87.3 words
Max summary length: 95 words
Min summary length: 83 words


In [11]:
# Save results - one file per cluster
os.makedirs(OUTPUT_DIR, exist_ok=True)

saved_files = []
for result in tqdm(results, desc="Saving cluster files"):
    cluster_label = result['cluster_label']
    
    # Create individual cluster output
    cluster_output = {
        'cluster_label': cluster_label,
        'cluster_size': result['cluster_size'],
        'cluster_summary': result['cluster_summary'],
        'reasons': result['reasons'],
        # 'original_indices': result['original_indices']
    }
    
    # Save as JSON
    output_file = os.path.join(OUTPUT_DIR, f"cluster_{cluster_label}_analysis.json")
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(cluster_output, f, indent=2, ensure_ascii=False)
    
    saved_files.append(output_file)

# Also save summary CSV for overview
summary_df_save = results_df[['cluster_label', 'cluster_size', 'cluster_summary', 'reasons', 'round1_cluster_labels']].copy()
summary_csv = os.path.join(OUTPUT_DIR, f"cluster_summaries_overview.csv")
summary_df_save.to_csv(summary_csv, index=False)

print(f"\n✓ Saved {len(saved_files)} individual cluster files to: {OUTPUT_DIR}")
print(f"  Files: cluster_0_analysis.json, cluster_1_analysis.json, ...")
print(f"\n✓ Saved summary overview to: {summary_csv}")

Saving cluster files:   0%|          | 0/10 [00:00<?, ?it/s]


✓ Saved 10 individual cluster files to: results/cluster_analysis_layer3
  Files: cluster_0_analysis.json, cluster_1_analysis.json, ...

✓ Saved summary overview to: results/cluster_analysis_layer3/cluster_summaries_overview.csv
